# Limpieza de datos - Clustering de Usuarios

Este notebook realiza la limpieza de datos específica para el **proyecto de clustering de usuarios**. A diferencia del proyecto de regresión, aquí:

- ✅ **CustomerID nulos**: Se ELIMINAN (requisito obligatorio - sin cliente no hay cluster)
- ✅ **Description nulas**: Se ELIMINAN (sin descripción no hay info del producto)
- ✅ **Cancelaciones**: Se CONSERVAN (la tasa de devolución es un patrón de comportamiento)
- ✅ **Outliers**: Capping al p99 (balance entre mantener datos y reducir extremos)

## Anomalías identificadas en el EDA:
- Registros duplicados
- Transacciones sin identificación de usuario (CustomerID nulo)
- Cantidades y precios unitarios negativos o cero
- Outliers en cantidades y precios unitarios
- StockCodes no estándar (códigos operacionales)

## 0. Imports y Carga del Dataset

Preparamos el entorno de trabajo: importamos librerías, configuramos la estética visual, definimos las rutas del proyecto y cargamos el dataset original. 

A continuación creamos las columnas auxiliares necesarias para los pasos de limpieza:
- `Fecha`: fecha sin hora (normalizada)
- `Mes`: periodo mensual
- `DiaSemana`: nombre del día de la semana
- `EsCancelacion`: booleano si el InvoiceNo empieza con 'C'
- `TotalPrice`: cantidad × precio unitario

Inicializamos el diccionario de auditoría `stats_cleaning` que registrará paso a paso cuántos registros eliminamos y por qué.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración visual global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
%matplotlib inline

# Rutas del proyecto
RUTA_CSV      = '../../../data/raw/data.csv'
RUTA_GRAFICOS = '../../../graphics/clean/'
RUTA_INTERIM  = '../../../data/interim/interim_ProyClustering/'
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
os.makedirs(RUTA_INTERIM,  exist_ok=True)

print('Librerías cargadas correctamente.')
print(f'Rutas configuradas:')
print(f'  - CSV raw    : {RUTA_CSV}')
print(f'  - Gráficos   : {RUTA_GRAFICOS}')
print(f'  - Interim    : {RUTA_INTERIM}')

Librerías cargadas correctamente.
Rutas configuradas:
  - CSV raw    : ../../../data/raw/data.csv
  - Gráficos   : ../../../graphics/clean/
  - Interim    : ../../../data/interim/interim_ProyClustering/


In [2]:
# Carga del dataset original
df_raw     = pd.read_csv(RUTA_CSV, encoding='latin-1')
df_working = df_raw.copy()

# ── Columnas auxiliares ────────────────────────────────────────────────────────
df_working['InvoiceDate']   = pd.to_datetime(df_working['InvoiceDate'], format='mixed')
df_working['Fecha']         = df_working['InvoiceDate'].dt.normalize()
df_working['Mes']           = df_working['InvoiceDate'].dt.to_period('M')
df_working['DiaSemana']     = df_working['InvoiceDate'].dt.day_name()
df_working['EsCancelacion'] = df_working['InvoiceNo'].str.startswith('C')
df_working['TotalPrice']    = df_working['Quantity'] * df_working['UnitPrice']

# ── Diccionario de auditoría (se actualiza en cada paso) ──────────────────────
stats_cleaning = {'Registros Iniciales': len(df_raw)}

# ── Resumen de carga ──────────────────────────────────────────────────────────
print('=' * 55)
print(f'  DATASET CARGADO - CLUSTERING DE USUARIOS')
print('=' * 55)
print(f'  Filas    : {df_raw.shape[0]:>10,}')
print(f'  Columnas : {df_raw.shape[1]:>10}')
print('=' * 55)
print(f'\n  df_working activo : {len(df_working):,} filas')
print(f'  Columnas auxiliares añadidas:')
print(f'    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice')
print(f'\n  Columnas originales: {list(df_raw.columns)}')

  DATASET CARGADO - CLUSTERING DE USUARIOS
  Filas    :    541,909
  Columnas :          8

  df_working activo : 541,909 filas
  Columnas auxiliares añadidas:
    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice

  Columnas originales: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [3]:
# ── Inspección inicial de valores nulos ───────────────────────────────────────
print('\n' + '=' * 55)
print('  VALORES NULOS INICIALES')
print('=' * 55)

nulos_count = df_working.isnull().sum()
nulos_pct = (df_working.isnull().sum() / len(df_working) * 100).round(2)
nulos_df = pd.DataFrame({
    'Nulos': nulos_count,
    'Porcentaje': nulos_pct
})
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Nulos', ascending=False)

if len(nulos_df) > 0:
    print(f'\n  Columnas con valores nulos:')
    for col, row in nulos_df.iterrows():
        print(f"    {col:<15} : {row['Nulos']:>10,} ({row['Porcentaje']:>6.2f}%)")
else:
    print(f'\n  ✓ No hay valores nulos en el dataset')

print(f'\n  ⚠ CRÍTICO para clustering: CustomerID tiene {nulos_df.loc["CustomerID", "Nulos"] if "CustomerID" in nulos_df.index else 0:,} nulos → DEBEN ser eliminados')


  VALORES NULOS INICIALES

  Columnas con valores nulos:
    CustomerID      :  135,080.0 ( 24.93%)
    Description     :    1,454.0 (  0.27%)

  ⚠ CRÍTICO para clustering: CustomerID tiene 135,080 nulos → DEBEN ser eliminados
